# Leakage-Safe METR-LA Preprocessing

**UrbanFlow AI - Task 03**

Preprocessing turns the verified traffic table into chronological, model-ready forecasting examples. The central rule is that information from validation or test periods must never influence training-fitted statistics. This notebook demonstrates the reusable implementation; it does not train a forecasting model.

## 1. Load the verified METR-LA dataset

The raw CSV is preserved unchanged. The trusted graph pickle is loaded only from the documented local path, and sensor order is checked again before arrays are created.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw' / 'metr-la').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'raw' / 'metr-la').exists():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks directory.')
sys.path.insert(0, str(ROOT))

from src.data.inspection import load_graph_bundle, load_speed_data, read_sensor_ids, validate_graph_alignment
from src.data.preprocessing import (
    INPUT_ZERO_POLICY, PerSensorStandardizer, all_sensor_zero_mask, chronological_split_ranges,
    cyclical_time_of_day, measurement_masks, prepare_speed_inputs, target_evaluation_mask,
)
from src.data.windows import build_forecast_windows, window_time_features

DATA_DIR = ROOT / 'data' / 'raw' / 'metr-la'
speed_frame = load_speed_data(DATA_DIR / 'METR-LA.csv')
graph = load_graph_bundle(DATA_DIR / 'adj_mx_METR-LA.pkl')
reference_ids = read_sensor_ids(DATA_DIR / 'graph_sensor_ids.txt')
alignment = validate_graph_alignment(speed_frame.columns, graph, reference_ids)
required = ['csv_graph_same_order', 'reference_graph_same_order', 'mapping_matches_graph_order', 'adjacency_shape_matches_ids']
assert all(alignment[key] for key in required)
speeds = speed_frame.to_numpy(dtype=np.float64, copy=False)
display(pd.Series({'rows': speeds.shape[0], 'sensors': speeds.shape[1], 'start': speed_frame.index[0], 'end': speed_frame.index[-1], 'adjacency_shape': graph.adjacency.shape}).to_frame('value'))

,value
rows,34272
sensors,207
start,2012-03-01 00:00:00
end,2012-06-27 23:55:00
adjacency_shape,"(207, 207)"


## 2. Chronological train/validation/test splitting

Rows are never shuffled. The initial 70%/10%/20% split uses half-open row ranges: the start row is included and the stop row is excluded. Validation and test inputs may use the immediately preceding 60 minutes because those observations were available at the forecast origin, but their targets must stay inside their own partition.

In [2]:
splits = chronological_split_ranges(speed_frame.index)
split_table = pd.DataFrame([
    {'split': name, 'row_range': f'[{s.start}, {s.stop})', 'rows': s.count, 'start_timestamp': s.start_timestamp, 'end_timestamp': s.end_timestamp}
    for name, s in splits.items()
]).set_index('split')
display(split_table)

,row_range,rows,start_timestamp,end_timestamp
split,,,,
train,"[0, 23990)",23990,2012-03-01 00:00:00,2012-05-23 07:05:00
validation,"[23990, 27417)",3427,2012-05-23 07:10:00,2012-06-04 04:40:00
test,"[27417, 34272)",6855,2012-06-04 04:45:00,2012-06-27 23:55:00


## 3. Build 12-input/12-output forecasting windows

Each input contains 12 five-minute speed observations with shape `(12, 207)`. Its target is the following 12 observations. The forecast origin is the last input timestamp, so target steps 3, 6 and 12 are exactly +15, +30 and +60 minutes. Sliding views avoid unnecessary copies of the large speed array.

In [3]:
window_sets = {name: build_forecast_windows(speeds, speed_frame.index, split) for name, split in splits.items()}
window_table = pd.DataFrame([
    {'split': name, 'samples': len(w.X_speed), 'X_speed': str(w.X_speed.shape), 'y_speed': str(w.y_speed.shape), 'first_origin': w.forecast_origin_timestamps[0], 'last_target': w.target_timestamps[-1, -1]}
    for name, w in window_sets.items()
]).set_index('split')
display(window_table)

example = window_sets['validation']
horizon_check = pd.Series({
    '+15 minutes (step 3)': example.target_timestamps[0, 2] - example.forecast_origin_timestamps[0],
    '+30 minutes (step 6)': example.target_timestamps[0, 5] - example.forecast_origin_timestamps[0],
    '+60 minutes (step 12)': example.target_timestamps[0, 11] - example.forecast_origin_timestamps[0],
})
display(horizon_check.to_frame('observed_offset'))

,samples,X_speed,y_speed,first_origin,last_target
split,,,,,
train,23967,"(23967, 12, 207)","(23967, 12, 207)",2012-03-01 00:55:00,2012-05-23 07:05:00
validation,3416,"(3416, 12, 207)","(3416, 12, 207)",2012-05-23 07:05:00,2012-06-04 04:40:00
test,6844,"(6844, 12, 207)","(6844, 12, 207)",2012-06-04 04:40:00,2012-06-27 23:55:00


,observed_offset
+15 minutes (step 3),0 days 00:15:00
+30 minutes (step 6),0 days 00:30:00
+60 minutes (step 12),0 days 01:00:00


## 4. Fit normalization on training observations only

A separate mean and standard deviation are fitted for each sensor because sensors have different normal speed levels. Exact zeros and non-finite values are excluded from these fitted statistics: the dataset does not establish that synchronized zeros are genuine traffic speeds. The raw zeros are not replaced or deleted; when transformed they remain the standardized representation of recorded 0 mph. The same stored parameters are applied unchanged to every split and can be inverted back to mph.

In [4]:
train = splits['train']
standardizer = PerSensorStandardizer.fit(speeds[train.start:train.stop], exclude_zeros=True)
normalization_summary = pd.Series({
    'method': 'per-sensor standardization',
    'fit rows': f'[{train.start}, {train.stop}) only',
    'zero policy for fitted statistics': 'exclude exact zeros',
    'minimum sensor mean (mph)': standardizer.mean.min(),
    'maximum sensor mean (mph)': standardizer.mean.max(),
    'minimum sensor scale (mph)': standardizer.scale.min(),
    'maximum sensor scale (mph)': standardizer.scale.max(),
})
display(normalization_summary.to_frame('value'))
sample = speeds[:24]
reconstructed = standardizer.inverse_transform(standardizer.transform(sample))
print('Maximum inverse-transform error (mph):', np.max(np.abs(reconstructed - sample)))

,value
method,per-sensor standardization
fit rows,"[0, 23990) only"
zero policy for fitted statistics,exclude exact zeros
minimum sensor mean (mph),30.874076
maximum sensor mean (mph),67.762287
minimum sensor scale (mph),2.032602
maximum sensor scale (mph),23.586636


Maximum inverse-transform error (mph): 7.105427357601002e-15


## 5. Preserve zeros and create explicit validity masks

Zeros remain in model inputs, accompanied by explicit masks so later models can distinguish them. No future value is used to fill an input. The proposed primary evaluation mask excludes non-finite and exact-zero targets consistently for every model. This avoids scoring long synchronized zero periods as ordinary traffic, but it may also exclude genuine stopped traffic; results should later include a sensitivity check that retains finite zeros.

In [5]:
raw_masks = measurement_masks(speeds)
zero_summary = []
horizons = {'+15': 2, '+30': 5, '+60': 11}
for name, split in splits.items():
    targets = window_sets[name].y_speed
    valid = target_evaluation_mask(targets, exclude_zeros=True)
    row = {
        'split': name,
        'all target values valid (%)': 100 * valid.mean(),
        'all-sensor-zero timestamps': int(all_sensor_zero_mask(speeds[split.start:split.stop]).sum()),
    }
    row.update({f'{label} valid (%)': 100 * valid[:, step].mean() for label, step in horizons.items()})
    zero_summary.append(row)
display(pd.DataFrame(zero_summary).set_index('split').round(3))
print('Recorded exact-zero cells are unchanged:', int(raw_masks['zero'].sum()))
print('Input-zero policy:', INPUT_ZERO_POLICY)

# Deterministic extreme cases make the policy and failure behavior visible.
toy_scaler = PerSensorStandardizer.fit(np.array([[10.0, 20.0], [12.0, 24.0]]))
all_zero_window = np.zeros((1, 12, 2))
standardized_zero_window, zero_window_masks = prepare_speed_inputs(all_zero_window, toy_scaler)
assert zero_window_masks['zero'].all()
np.testing.assert_allclose(toy_scaler.inverse_transform(standardized_zero_window), all_zero_window, atol=1e-12)
print('All-zero input window preserved and fully zero-masked:', zero_window_masks['zero'].all())

try:
    PerSensorStandardizer.fit(np.zeros((3, 1)))
except ValueError as error:
    print('No-valid-training-reading check:', error)

validation_windows = window_sets['validation']
assert validation_windows.input_start_rows[0] == splits['validation'].start - 12
assert validation_windows.target_start_rows[0] == splits['validation'].start
print('Boundary check: validation context starts before the split, while its first target starts at the split.')

,all target values valid (%),all-sensor-zero timestamps,+15 valid (%),+30 valid (%),+60 valid (%)
split,,,,,
train,92.870,1301,92.870,92.870,92.870
validation,93.019,181,93.019,93.018,93.019
test,87.881,666,87.883,87.881,87.879


Recorded exact-zero cells are unchanged: 575302
Input-zero policy: preserve_recorded_zeros_with_masks
All-zero input window preserved and fully zero-masked: True
No-valid-training-reading check: No valid training observations for sensor indices [0]
Boundary check: validation context starts before the split, while its first target starts at the split.


## 6. Add cyclical time-of-day features

Sine and cosine encode the 24-hour cycle without learning from traffic observations. Historical features align with the 12 input timestamps. Known-future calendar features align with target timestamps and are kept separate because they are known at prediction time, unlike future traffic speeds.

In [6]:
time_of_day = cyclical_time_of_day(speed_frame.index)
time_shapes = []
for name, windows in window_sets.items():
    historical_time, known_future_time = window_time_features(time_of_day, windows)
    time_shapes.append({'split': name, 'historical_time': str(historical_time.shape), 'known_future_time': str(known_future_time.shape)})
display(pd.DataFrame(time_shapes).set_index('split'))
display(pd.DataFrame(time_of_day[:4], index=speed_frame.index[:4], columns=['time_sin', 'time_cos']))

,historical_time,known_future_time
split,,
train,"(23967, 12, 2)","(23967, 12, 2)"
validation,"(3416, 12, 2)","(3416, 12, 2)"
test,"(6844, 12, 2)","(6844, 12, 2)"


,time_sin,time_cos
timestamp,,
2012-03-01 00:00:00,0.000000,1.000000
2012-03-01 00:05:00,0.021815,0.999762
2012-03-01 00:10:00,0.043619,0.999048
2012-03-01 00:15:00,0.065403,0.997859


## 7. Final preprocessing summary

The reusable pipeline now provides chronological row ranges and timestamps, leakage-safe windows, training-only per-sensor normalization, reversible mph conversion, explicit zero/validity masks, time-of-day features, and verified graph-node ordering. It deliberately does not decide that every zero is missing or genuine congestion. Task 04 should consume these same target timestamps and masks for both simple baselines so later comparisons remain fair.

In [7]:
assert speeds.shape == (34272, 207)
assert graph.adjacency.shape == (207, 207)
for name, windows in window_sets.items():
    split = splits[name]
    assert windows.X_speed.shape[1:] == (12, 207)
    assert windows.y_speed.shape[1:] == (12, 207)
    assert windows.target_start_rows.min() >= split.start
    assert (windows.target_start_rows + 12).max() <= split.stop
print('All notebook preprocessing assertions passed.')

All notebook preprocessing assertions passed.
